In [1]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("sunilthite/text-document-classification-dataset")

# print("Path to dataset files:", path)

In [2]:
import pandas as pd

In [3]:
# Load dataset and remove missing values from relevant columns
df = pd.read_csv("./data/df_file.csv")   
df = df.dropna()

df.head()

,Text,Label
0,Budget to set scene for election\n \n Gordon B...,0
1,Army chiefs in regiments decision\n \n Militar...,0
2,Howard denies split over ID cards\n \n Michael...,0
3,Observers to monitor UK election\n \n Minister...,0
4,Kilroy names election seat target\n \n Ex-chat...,0


In [4]:
# Clean text data
import re

def clean_text(t):
    t = t.lower()
    t = re.sub(r'http\S+', '', t)
    t = re.sub(r'[^a-zA-Z0-9\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


In [5]:
# Apply preprocessing
df["Clean_Text"] = df["Text"].apply(clean_text)
df.head()

,Text,Label,Clean_Text
0,Budget to set scene for election\n \n Gordon B...,0,budget to set scene for election gordon brown ...
1,Army chiefs in regiments decision\n \n Militar...,0,army chiefs in regiments decision military chi...
2,Howard denies split over ID cards\n \n Michael...,0,howard denies split over id cards michael howa...
3,Observers to monitor UK election\n \n Minister...,0,observers to monitor uk election ministers wil...
4,Kilroy names election seat target\n \n Ex-chat...,0,kilroy names election seat target exchat show ...


In [ ]:
# Convert text to embeddings (BERT)
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-mpnet-base-v2")

embeddings = model.encode(df["Clean_Text"].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings)
embeddings.shape


/opt/anaconda3/envs/ml_tutorial_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 70/70 [01:03<00:00,  1.10it/s]


(2225, 768)

: 

In [ ]:
# Determine optimal number of clusters using Silhouette Score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scores = []
K = range(2, 15)

for k in K:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(embeddings)
    score = silhouette_score(embeddings, km.labels_)
    scores.append(score)
    print(f"k={k} — Silhouette: {score:.4f}")



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
# Plot Silhouette Scores
import matplotlib.pyplot as plt

plt.plot(K, scores, marker="o")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Analysis")
plt.show()


In [ ]:
# Run final KMeans with optimal k = 8
k = 8

final_kmeans = KMeans(n_clusters=k, random_state=42)
df["cluster"] = final_kmeans.fit_predict(embeddings)

df.head()

In [ ]:
from sklearn.manifold import TSNE # ( t-Distributed Stochastic Neighbor Embedding ) , important for dimensionality reduction to visualize high-dimensional data in 2D or 3D space.

tsne = TSNE(n_components=2, perplexity=40, random_state=42)
tsne_results = tsne.fit_transform(embeddings)

df["x"] = tsne_results[:, 0]
df["y"] = tsne_results[:, 1]


In [ ]:
# # Visualize clusters using t-SNE

import seaborn as sns

plt.figure(figsize=(10, 7))
sns.scatterplot(data=df, x="x", y="y", hue="cluster", palette="tab10", s=40)
plt.title("t-SNE Visualization of Text Clusters")
plt.show()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer(stop_words="english", max_features=10000)
X_tfidf = vect.fit_transform(df["clean"])

terms = vect.get_feature_names_out()
clusters = df["cluster"].unique()

for c in clusters:
    idx = df[df["cluster"] == c].index
    tfidf_means = X_tfidf[idx].mean(axis=0).A1
    top = tfidf_means.argsort()[-15:][::-1]
    print(f"\n🔹 Cluster {c}:")
    print(", ".join(terms[i] for i in top))
